# 04 — Document Parsing with `ai_parse_document`

## Objective

Understand how an unstructured binary file (a PDF) becomes usable structured content, using Databricks' `ai_parse_document` AI Function. Notebooks 02-03 only ever produced `.txt`/`.csv` files; this notebook works with a real, synthetic **PDF** -- with a title, headings, paragraphs, and a table -- so parsing has something genuinely unstructured to work on.

## What We Will Learn

- How to get a file into a Unity Catalog Volume when the file wasn't created inside Databricks itself
- How `ai_parse_document` turns raw PDF bytes into structured output (pages, headings, paragraphs, tables)
- Why you should inspect the *actual* returned schema in your workspace rather than trust a fixed field list -- this function's output shape can evolve, and this notebook has not been executed against a live workspace
- How to distinguish a genuine parsing result from a fabricated one

## Prerequisites

- Completed `02_synthetic_data_generation.ipynb` and `03_document_ingestion_basics.ipynb` with matching catalog/schema/volume widget values
- The pre-generated PDF at `pdfs/16_fraud_escalation_playbook.pdf` in this repo, uploaded into your `<volume>/documents/` folder -- Step 2 below has exact instructions
- `ai_parse_document` is a **preview / entitlement-gated AI Function** -- it requires Unity Catalog, a supported Databricks Runtime, and availability in your workspace's region. If it's not enabled for you, Step 3 below will fail with a clear function-not-found or permission error; see **Common Errors / Limitations**

## Conceptual Explanation

**Why parsing is its own step.** Notebook 03 showed that `binaryFile` gives you raw bytes with no understanding of what's inside them. A PDF's bytes encode fonts, positions, and drawing instructions -- not "paragraph 2 of page 1". Parsing is the step that turns those bytes into a structure you can actually reason about: pages, headings, paragraphs, tables.

**`ai_parse_document`.** This is a Databricks-native AI Function, callable from SQL or Python, that takes binary document content (PDF, and other supported formats) and returns a structured representation -- conceptually: a list of pages, each containing elements such as titles, paragraphs, and tables. It runs as a managed service call, not a Python library you import, which is why it needs to be enabled/entitled in your workspace.

**Why this PDF wasn't generated inside this notebook.** The original plan was to build the PDF in Databricks with `fpdf2`, same as everything else in this project. That hit a persistent `FPDFException: Not enough horizontal space to render a single character` in the Databricks Python environment -- it didn't go away even after switching APIs and passing explicit widths, while the exact same generation code ran cleanly outside Databricks (confirmed locally: a valid 1-page, ~1.9KB PDF). That points at something specific to that Databricks environment -- most likely an old `fpdf`/PyFPDF package shadowing `fpdf2` -- rather than a bug in the generation code. Rather than keep fighting an environment issue that's orthogonal to what this notebook teaches, the PDF was generated once, locally, and is uploaded manually instead. Once it's a file in a Volume, it's parsed exactly the same way regardless of where it came from.

**Why this notebook doesn't hardcode the output schema.** Per this project's own rules, we don't fabricate execution results or claim something was tested when it wasn't. `ai_parse_document`'s exact returned field names are a live product detail that can change between Databricks Runtime versions, and this notebook has not been run against a real workspace. So instead of asserting "the output has fields X, Y, Z", the implementation below shows you how to **inspect the real schema Databricks gives you**, and adapt from there. That's a more useful habit for any AI Function anyway, since most of them evolve over time.

**Parsing limitations (to expect, not to fear).** Parsers can merge or split paragraphs differently than a human would, mis-tag a heading as a paragraph (or vice versa), or struggle with complex nested tables. Treat parsed output as a strong starting point, not ground truth -- Notebook 12 (RAG failure cases) revisits this.

## Example Data

A one-page synthetic PDF, pre-generated with `fpdf2` and committed to this repo at `pdfs/16_fraud_escalation_playbook.pdf`: **"Fraud Escalation Playbook"** -- a fictional Aurora Trust Bank policy with a title, an overview paragraph, a numbered list of steps, and a 4-row escalation table. It deliberately includes a table so you can see whether the parser recognizes it as one.

## Implementation

### Step 1 — Point at the same catalog/schema/volume as Notebooks 02-03

In [ ]:
dbutils.widgets.text("catalog_name", "main", "Unity Catalog catalog")
dbutils.widgets.text("schema_name", "genai_lab", "Schema")
dbutils.widgets.text("volume_name", "synthetic_data", "Volume")

catalog_name = dbutils.widgets.get("catalog_name")
schema_name = dbutils.widgets.get("schema_name")
volume_name = dbutils.widgets.get("volume_name")
volume_path = f"/Volumes/{catalog_name}/{schema_name}/{volume_name}"
documents_path = f"{volume_path}/documents"
pdf_volume_path = f"{documents_path}/16_fraud_escalation_playbook.pdf"

print(f"Expected PDF path: {pdf_volume_path}")

### Step 2 — Upload the pre-generated PDF to your Volume

1. In the Databricks workspace, go to **Catalog** (left sidebar) > your catalog > your schema > **Volumes** > your volume > the `documents/` folder.
2. Click **Upload to this volume** and select `pdfs/16_fraud_escalation_playbook.pdf` from this repo.
   - Alternatively, with the Databricks CLI configured locally: `databricks fs cp pdfs/16_fraud_escalation_playbook.pdf dbfs:/Volumes/<catalog>/<schema>/<volume>/documents/16_fraud_escalation_playbook.pdf`
3. Run the cell below to confirm it landed at the expected path -- it fails loudly with a clear message if not.

In [ ]:
try:
    file_info = dbutils.fs.ls(pdf_volume_path)[0]
    print(f"Found: {file_info.path} ({file_info.size} bytes)")
except Exception as e:
    raise FileNotFoundError(
        f"Could not find {pdf_volume_path}. Upload pdfs/16_fraud_escalation_playbook.pdf from the "
        f"repo to this Volume path first -- see Step 2 above."
    ) from e

### Step 3 — Read it as binary, then parse it with `ai_parse_document`

This is the same `binaryFile` pattern from Notebook 03 -- `ai_parse_document` takes that raw binary `content` column as input. It doesn't matter that this file was uploaded rather than written by a previous cell; the bytes are the bytes.

In [ ]:
pdf_binary_df = spark.read.format("binaryFile").load(pdf_volume_path)
pdf_binary_df.createOrReplaceTempView("fraud_pdf_raw")
display(pdf_binary_df.select("path", "length", "modificationTime"))

In [ ]:
%sql
SELECT path, ai_parse_document(content) AS parsed
FROM fraud_pdf_raw

**Stop and look at the result above before writing any more code.** Expand the `parsed` column in the results grid -- that's the real, workspace-specific schema you'll work with. It will broadly contain something like a list of pages, each with elements tagged by type (heading, paragraph, table, ...), but the exact field names are a Databricks Runtime version detail we won't guess at here.

### Step 4 — Capture the result in Python and inspect its schema programmatically

Same query, but kept as a DataFrame so you can call `printSchema()` and look at one row directly -- this is the authoritative source of truth for field names in *your* workspace, not this notebook's prose.

In [ ]:
parsed_df = spark.sql("SELECT path, ai_parse_document(content) AS parsed FROM fraud_pdf_raw")
parsed_df.printSchema()

parsed_row = parsed_df.first()
parsed_value = parsed_row["parsed"]
print(type(parsed_value))
print(parsed_value)

If `parsed_value` prints as a JSON string, the cell below pretty-prints it so it's easier to read and find the field names to use next. If it's already a native struct, `printSchema()` above already told you the field names to select.

In [ ]:
import json

if isinstance(parsed_value, str):
    try:
        print(json.dumps(json.loads(parsed_value), indent=2))
    except json.JSONDecodeError:
        print("parsed_value is a string but not valid JSON -- inspect it as-is:")
        print(parsed_value)
else:
    print("parsed_value is already a structured type -- use parsed_df.printSchema() above to navigate it.")

## Inspect the Output

- Does the parsed result separate the title ("Fraud Escalation Playbook") from the body paragraphs?
- Does it recognize the **Escalation Tiers** table as tabular data, or flatten it into plain text? This is the single most useful thing to check -- table handling is where parsers vary the most.
- Does the numbered "Escalation Steps" list come through as four separate elements, or one merged block of text?
- Compare page/element counts against what you know is actually in the PDF (1 page; 1 title; 1 overview paragraph; 4 steps; 1 table) -- any mismatch tells you something concrete about this parser's behavior, not a guess.

## Experimentation Section

1. Open `pdfs/16_fraud_escalation_playbook.pdf` locally in a PDF viewer and compare it side by side with what `ai_parse_document` returned -- easiest way to spot anything the parser dropped or reordered.
2. Rename the uploaded file and update `pdf_volume_path` accordingly -- confirms Step 2's failure message actually triggers correctly when the path is wrong.
3. If you do want to try generating a PDF directly in Databricks again: install `fpdf2` in a notebook cell, then immediately run `import fpdf; print(fpdf.__file__, getattr(fpdf, "__version__", "NOT FOUND"))` -- if the version is missing or the path looks like a cluster-level library rather than your notebook's environment, that confirms the shadowing theory from **Conceptual Explanation**.
4. Generate a second, different synthetic PDF locally (e.g. a 2-page document, or one without a table) and upload it the same way -- compare how `ai_parse_document` handles it.
5. Convert one of the 15 `.txt` documents from Notebook 02 into a PDF locally, upload it, parse it, and compare the parser's output against the original known-good text -- does anything get dropped or reordered?

## Common Errors / Limitations

- **File not found at the expected Volume path** -- Step 2's upload didn't complete, or `catalog_name`/`schema_name`/`volume_name` don't match where you uploaded it. The Step 2 check cell gives you the exact expected path.
- **`UNRESOLVED_ROUTINE` / function not found** -- `ai_parse_document` isn't enabled for your workspace, region, or SQL warehouse/cluster type. Check your Databricks release notes or ask a workspace admin whether AI Functions / Mosaic AI document parsing is available on your plan.
- **Permission or entitlement errors** -- separate from "not found": the function exists but your workspace isn't entitled to call it. Same remedy -- check with an admin.
- **Do not assume the parsed schema shown in this notebook's prose is exact** -- we deliberately did not hardcode field-access code here, because this notebook was authored without running against a live workspace. Always verify with `printSchema()` / `DESCRIBE FUNCTION ai_parse_document` first, as Step 4 does.
- **Parsing cost and latency** -- each call is a managed AI service invocation, not free local computation. Don't loop `ai_parse_document` over a large batch of documents without first confirming cost/throughput expectations for your workspace.
- **If you regenerate the PDF locally**, re-run `python <script> pdfs/16_fraud_escalation_playbook.pdf` (or your own script) and re-upload -- Step 2's Databricks-side check only verifies a file exists at the path, not that its content matches this notebook's description.

## Summary

You uploaded a pre-generated PDF into a Unity Catalog Volume, read it as raw binary (the same representation from Notebook 03), and ran it through `ai_parse_document`. Rather than trusting a hardcoded output schema, you inspected the real one your workspace returned -- a habit worth keeping for every AI Function in this project. You also saw a concrete case where the pragmatic move was to sidestep an environment issue (PDF generation inside Databricks) rather than debug it indefinitely, since it wasn't actually what this notebook set out to teach.

## Suggested Exercises

- Write down, in your own words, the exact field path you'd use to get from `parsed` down to "just the table rows" for *your* workspace's schema.
- Compare the parsed PDF text against the hand-written `.txt` files from Notebook 02 -- which was easier to get clean text from, and why?
- When you're ready, move on to **`05_document_classification_ai_classify.ipynb`**, which classifies documents (including this new one) into categories using `ai_classify`.